# 04 — Análise estatística (Nível 1 — estado de São Paulo)

**Objetivo:** testar a hipótese central do estudo:

> Municípios de SP com maior proporção de idosos morando sozinhos têm maior
> taxa de internação por causas associadas a falta de socorro imediato
> (lesões/causas externas, sintomas mal definidos, transtornos mentais),
> mesmo controlando pelo IDH municipal?

**Entrada:** `data/processed/dataset_municipios_sp.csv` — a versão **limpa**
do notebook 03 (sem outliers, só municípios com IDH). A versão bruta
(`dataset_municipios_sp_bruto.csv`, 645 municípios) entra na checagem de
robustez da seção 4.4.

**Saídas:** figuras em `outputs/figures/` e resultados em `outputs/tables/`
— prontos para a seção de Resultados do artigo.


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..") / "src"))
import config  # noqa: E402

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

sns.set_theme(style="whitegrid")

mun = pd.read_csv(config.DATA_PROCESSED / "dataset_municipios_sp.csv")
bruto = pd.read_csv(config.DATA_PROCESSED / "dataset_municipios_sp_bruto.csv")
bruto_com_idh = bruto.dropna(subset=["idhm"]).copy()

x = "pct_idosos_sozinhos"
y = "taxa_internacao_100k_domicilios_idosos"

print(f"base limpa: {mun.shape} | base bruta com IDH: {bruto_com_idh.shape}")
mun.head()


## 4.1 Estatística descritiva


In [ ]:
mun[[x, y, "idhm", "domicilios_resp_idoso", "internacoes_total"]].describe().T


## 4.2 Correlação: % de idosos sozinhos × taxa de internação

Um ponto por município.


In [ ]:
r, p = stats.pearsonr(mun[x], mun[y])
print(f"Correlação de Pearson (n={len(mun)}): r={r:.3f}, p={p:.4g}")

fig, ax = plt.subplots(figsize=(8, 6))
sns.regplot(data=mun, x=x, y=y, ax=ax, scatter_kws={"alpha": 0.5})

rc = mun[mun["municipio"] == config.RIO_CLARO_NOME]
if not rc.empty:
    ax.scatter(rc[x], rc[y], color="red", s=100, zorder=5, label=config.RIO_CLARO_NOME)
    ax.legend()

ax.set_xlabel("% de domicílios com responsável idoso que são unipessoais")
ax.set_ylabel("Internações por 100 mil domicílios com responsável idoso")
ax.set_title(f"Idosos sozinhos × internações — municípios de SP (n={len(mun)})")
fig.tight_layout()
fig.savefig(config.OUTPUTS_FIGURES / "correlacao_idosos_sozinhos_internacoes.png", dpi=150)
plt.show()


## 4.3 Modelo principal — regressão controlando por IDH

Modelo: `taxa_internacao_100k_domicilios_idosos ~ pct_idosos_sozinhos + idhm`

O IDH entra como controle porque municípios mais pobres tendem a internar
mais por razões que nada têm a ver com morar sozinho. O coeficiente de
`pct_idosos_sozinhos` é o teste central do estudo.


In [ ]:
modelo = smf.ols(f"{y} ~ {x} + idhm", data=mun).fit()
print(modelo.summary())

with open(config.OUTPUTS_TABLES / "regressao_ols.txt", "w") as f:
    f.write(modelo.summary().as_text())


**Leitura do modelo principal:**

- `pct_idosos_sozinhos`: coeficiente **positivo**, na direção prevista pela
  hipótese, significativo no limiar convencional de 5%.
- `idhm`: coeficiente **negativo e fortemente significativo** — municípios com
  IDH maior internam menos idosos. Faz sentido e serve como validação de
  sanidade (se tivesse dado o contrário, haveria algo errado no dado).
- **R² baixo** (~0,08): as duas variáveis explicam uma fração pequena da
  variação entre municípios. Esperado num estudo ecológico — internação
  depende de muitos outros fatores (oferta de leitos, perfil etário fino,
  cobertura de atenção básica, prevalência de crônicas). O achado é sobre
  *existir associação na direção prevista*, não sobre morar sozinho explicar
  a internação sozinho.

⚠️ **Não pare aqui.** Este resultado isolado é frágil. A seção 4.4 testa se
ele sobrevive a outras especificações — e a resposta muda bastante o que dá
para afirmar no artigo.


## 4.4 Checagem de robustez — o resultado sobrevive a outras especificações?

Um coeficiente com p logo abaixo de 0,05 numa única especificação é o tipo de
achado que um parecerista derruba. Aqui rodamos o mesmo teste de seis
maneiras diferentes e olhamos se o sinal se mantém:

1. **OLS (principal)** — o modelo da seção 4.3.
2. **OLS com erros-padrão robustos (HC3)** — não assume variância constante
   dos resíduos, que raramente vale em dados municipais.
3. **OLS + porte do município** — municípios pequenos têm taxa por 100 mil
   mais alta *e* mais idosos sozinhos; sem esse controle, parte do efeito
   pode ser só tamanho.
4. **OLS na base bruta** (645 municípios com IDH, sem remover outliers).
5. **Binomial negativa** com offset — modelo de contagem, padrão em
   epidemiologia para taxas de internação; trata a superdispersão.
6. **Poisson com erros robustos** — também de contagem, mas pondera cada
   município pelo volume de internações (municípios grandes pesam muito mais).


In [ ]:
for d in (mun, bruto_com_idh):
    d["log_porte"] = np.log10(d["domicilios_resp_idoso"])

resultados = []


def registrar(rotulo, modelo, escala, n):
    resultados.append({
        "Especificação": rotulo,
        "n": n,
        "coef": modelo.params[x],
        "p-valor": modelo.pvalues[x],
        "Direção": "positiva" if modelo.params[x] > 0 else "NEGATIVA",
        "p<0,05": "sim" if modelo.pvalues[x] < 0.05 else "não",
        "escala": escala,
    })


registrar("1. OLS (principal)", modelo, "internações/100 mil", len(mun))
registrar("2. OLS, erros robustos HC3",
          smf.ols(f"{y} ~ {x} + idhm", data=mun).fit(cov_type="HC3"), "internações/100 mil", len(mun))
registrar("3. OLS + porte do município",
          smf.ols(f"{y} ~ {x} + idhm + log_porte", data=mun).fit(), "internações/100 mil", len(mun))
registrar("4. OLS na base bruta",
          smf.ols(f"{y} ~ {x} + idhm + log_porte", data=bruto_com_idh).fit(),
          "internações/100 mil", len(bruto_com_idh))
registrar("5. Binomial negativa (offset)",
          smf.negativebinomial(f"internacoes_total ~ {x} + idhm", data=mun,
                               offset=np.log(mun["domicilios_resp_idoso"])).fit(disp=0),
          "log da taxa", len(mun))
registrar("6. Poisson robusto (offset)",
          smf.glm(f"internacoes_total ~ {x} + idhm", data=mun, family=sm.families.Poisson(),
                  offset=np.log(mun["domicilios_resp_idoso"])).fit(cov_type="HC1"),
          "log da taxa", len(mun))

robustez = pd.DataFrame(resultados)
robustez["coef"] = robustez["coef"].round(4)
robustez["p-valor"] = robustez["p-valor"].round(4)
robustez.to_csv(config.OUTPUTS_TABLES / "robustez_especificacoes.csv", index=False)
print(robustez.to_string(index=False))


### O que esta tabela diz

O resultado **não é robusto**. Das seis especificações:

- Quatro dão coeficiente positivo (direção da hipótese), mas só parte delas
  cruza o limiar de 5%.
- Duas perdem significância (erros robustos, controle de porte) — ficam
  logo acima de 0,05, o que é "sugestivo" e nada mais.
- O modelo **Poisson inverte o sinal**. Não é erro de código: esse modelo
  pondera cada município pelo volume de internações, e os 5 maiores
  concentram ~38% de todas as internações da base. Ou seja, ele mede
  essencialmente o padrão *entre as grandes cidades*, que é diferente do
  padrão geral. A binomial negativa, que desconta a superdispersão e
  equilibra os pesos, volta a dar positivo.

**Conclusão honesta para o artigo:** há evidência **fraca e na direção
prevista** de associação entre proporção de idosos morando sozinhos e taxa de
internação, sensível à forma de modelar. Isso é bem diferente de "não há
associação" (que era o resultado com o dado errado, por local de internação),
mas também não autoriza afirmar associação estabelecida.

A redação defensável é algo como: *"Observou-se associação positiva entre a
proporção de domicílios unipessoais com responsável idoso e a taxa de
internação municipal, controlada pelo IDH (β=171,8; p=0,042). A associação
mostrou-se sensível à especificação do modelo, perdendo significância sob
erros-padrão robustos e sob controle pelo porte populacional, o que indica
necessidade de cautela na interpretação."* — e então reportar a tabela inteira.

Isso é mais forte do que parece: um artigo que mostra a tabela de robustez
completa e discute a fragilidade passa por revisão muito melhor do que um que
reporta só a especificação favorável.


### 4.4b A associação muda conforme o porte do município?

A inversão do Poisson sugere que grandes e pequenos municípios podem se
comportar de forma diferente. Testamos isso formalmente com um termo de
interação.


In [ ]:
mun["pct_c"] = mun[x] - mun[x].mean()
mun["log_porte_c"] = mun["log_porte"] - mun["log_porte"].mean()

interacao = smf.ols(f"{y} ~ pct_c * log_porte_c + idhm", data=mun).fit()
print(interacao.summary().tables[1])

p_int = interacao.pvalues["pct_c:log_porte_c"]
print(f"\nInteração pct_idosos_sozinhos x porte: p={p_int:.3f}", "(significativa)" if p_int < 0.05 else "(NÃO significativa)")


O termo de interação **não é significativo**. Ou seja: embora as correlações
calculadas separadamente por faixa de porte pareçam diferentes, essa diferença
não se sustenta num teste formal — é compatível com variação amostral. **Não
afirmar moderação por porte no artigo.**

(Vale registrar a tentativa na Discussão como hipótese a investigar com mais
dados, mas não como resultado.)


## 4.5 Perfil das internações por causa

Lembrando: cada "causa" é um **capítulo da CID-10** (ver `config.CAUSAS_SIH`),
não o subgrupo específico do plano original.


In [ ]:
causas = list(config.CAUSAS_SIH.keys())
totais = bruto[causas].sum().rename(index=config.CAUSAS_SIH_LABELS)

fig, ax = plt.subplots(figsize=(7, 5))
totais.sort_values().plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_xlabel(f"Total de internações em idosos, {config.PERIODO_SIH} (estado de SP)")
ax.set_title("Perfil das internações em idosos, por capítulo CID-10")
fig.tight_layout()
fig.savefig(config.OUTPUTS_FIGURES / "perfil_causas.png", dpi=150)
plt.show()

print(totais.to_string())


## 4.6 Mapa coroplético (opcional — requer `geopandas` e o shapefile de SP)

1. Baixe: https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_municipais/municipio_2022/UFs/SP/SP_Municipios_2022.zip
2. Salve (sem descompactar) como `data/external/sp_municipios.zip`

O merge é por nome normalizado, como no resto do pipeline. Usa a base
**bruta** para o mapa ficar completo (645 municípios) — o mapa é descritivo,
não um teste de hipótese.


In [ ]:
try:
    import geopandas as gpd

    caminho_shp = config.DATA_EXTERNAL / "sp_municipios.zip"
    if caminho_shp.exists():
        gdf = gpd.read_file(f"zip://{caminho_shp}")
        gdf["municipio_norm"] = gdf["NM_MUN"].apply(config.normalizar_municipio)
        mapa = gdf.merge(bruto, on="municipio_norm", how="left")

        fig, ax = plt.subplots(figsize=(9, 9))
        mapa.plot(column=y, cmap="OrRd", legend=True, ax=ax, missing_kwds={"color": "lightgrey"})
        ax.set_title(f"Taxa de internação em idosos — SP ({config.PERIODO_SIH})")
        ax.axis("off")
        fig.tight_layout()
        fig.savefig(config.OUTPUTS_FIGURES / "mapa_taxa_internacao_sp.png", dpi=150)
        plt.show()
    else:
        print(f"Baixe o shapefile conforme instruções acima e salve em {caminho_shp}")
except ImportError:
    print("geopandas não instalado. No Anaconda Prompt: conda install -c conda-forge geopandas")
